# Lab 1: Kafka w Pythonie — producent, konsument, ETL w czasie rzeczywistym

## Część 1: Utwórz temat Kafka

In [1]:
! kafka-topics.sh --create --topic transactions --bootstrap-server broker:9092
! kafka-topics.sh --list --bootstrap-server broker:9092

Error while executing topic command : Topic 'transactions' already exists.
[2026-04-13 15:30:20,011] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'transactions' already exists.
 (org.apache.kafka.tools.TopicCommand)
__consumer_offsets
alerts
transactions


## Część 2: Producent — generowanie transakcji

#### zadanie 2.1

In [ ]:
from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']),
        'category': random.choice(['elektronika', 'odzież', 'żywność', 'książki']),
        'timestamp': datetime.now().isoformat(),
    }

while True:
    tx = generate_transaction()
    producer.send('transactions', value=tx)
    print(tx)
    time.sleep(1)

## Część 3: Konsument bezstanowy — filtrowanie

#### Zadanie 3.1 — Filtruj duże transakcje

In [ ]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Nasłuchuję na duże transakcje (amount > 3000)...")

for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        print(f"ALERT: {tx['tx_id']} | {tx['amount']:.2f} PLN | {tx['store']} | {tx['category']}")

#### Zadanie 3.2 — Transformacja i wzbogacenie

In [ ]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)


for message in consumer:
    tx = message.value

    if tx['amount'] > 3000:
        tx['risk_level'] = 'HIGH'
    elif tx['amount'] > 1000:
        tx['risk_level'] = 'MEDIUM'
    else:
        tx['risk_level'] = 'LOW'

    print(tx)

## Część 4: Konsument stanowy — agregacja

#### Zadanie 4.1 — Zliczanie transakcji per sklep

In [ ]:
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='count-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

#zliczanie transakcji

for message in consumer:
    tx = message.value

    store = tx['store']
    amount = tx['amount']

    # liczba transakcji
    store_counts[store] += 1

    # suma kwot
    if store not in total_amount:
        total_amount[store] = 0
    total_amount[store] += amount

    msg_count += 1

    #co 10 wiadomości napis z  podsumowaniem
    if msg_count % 10 == 0:
        print("\n--- PODSUMOWANIE ---")
        for s in store_counts:
            count = store_counts[s]
            total = total_amount[s]
            avg = total / count
            print(f"{s}: liczba={count}, suma={total:.2f}, średnia={avg:.2f}")

#### Zadanie 4.2 — Statystyki per kategoria

In [ ]:
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='stats-group-simple',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

stats = {}
msg_count = 0

#Zbieranie statystyk per kategoria

for message in consumer:
    tx = message.value
    category = tx['category']
    amount = tx['amount']

    if category not in stats:
        stats[category] = {
            'count': 0,
            'sum': 0.0,
            'min': amount,
            'max': amount
        }

    stats[category]['count'] += 1
    stats[category]['sum'] += amount

    if amount < stats[category]['min']:
        stats[category]['min'] = amount

    if amount > stats[category]['max']:
        stats[category]['max'] = amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("\n--- STATYSTYKI PER KATEGORIA ---")
        for cat in stats:
            print(
                f"{cat}: liczba={stats[cat]['count']}, "
                f"przychód={stats[cat]['sum']:.2f}, "
                f"min={stats[cat]['min']:.2f}, "
                f"max={stats[cat]['max']:.2f}"
            )

## Część 5: Wielu konsumentów

#### Zadanie 5.2 — Pytania

##### **1. Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?**
###### odp.Consumer odczyta zapisane wcześniej wiadomości z topicu, ponieważ Kafka przechowuje dane. Jeśli ustawione jest auto_offset_reset='earliest', to zacznie czytać od początku.
##### **2. Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?**
###### odp.Kafka rozdzieli obciążenie między konsumentów, przypisując im konkretne partycje topicu. Jeśli partycji jest więcej niż jedna, konsumenci będą pracować równolegle, natomiast jeśli jest tylko jedna, tylko jeden z nich będzie aktywny, co oznacza , że liczba pracujących konsumentów w grupie jest ograniczona przez liczbę partycji.
##### **3. Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?** 
###### odp.Przetwarzanie bezstanowe analizuje każdą wiadomość osobno. Przetwarzanie stanowe zapamiętuje dane i wykorzystuje je do obliczeń

## Praca domowa

In [ ]:
from kafka import KafkaConsumer
import json
from datetime import datetime
from collections import defaultdict

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    auto_offset_reset='earliest',
    group_id='anomaly-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

# przechowanie czasu transakcji dla każdego usera
user_transactions = defaultdict(list)


for message in consumer:
    tx = message.value
    user_id = tx['user_id']


    current_time = datetime.fromisoformat(tx['timestamp'])

    # dodanie nowej transakcji do listy usera
    user_transactions[user_id].append(current_time)

    # usunięckie starej transakcji, czyli starszej niż 60s
    user_transactions[user_id] = [
        t for t in user_transactions[user_id]
        if (current_time - t).total_seconds() <= 60
    ]


    if len(user_transactions[user_id]) > 3:
        print(f"ANOMALY: user={user_id} | tx_id={tx['tx_id']} | amount={tx['amount']}")